# Quick checks: `ods_alpha.scd1_mrc_pos_rent`

Отдельная тетрадка для 4 проверок:
1. Есть ли доступ к таблице.
2. Дубли по ключу `c_nmrc + d_rent`.
3. Конфликты, где у одного `c_nmrc + d_rent` разные `n_amt`.
4. Проверка за апрель для кейса `agr_id=413636181589`, `inn=2259000869`.

In [ ]:
import re
import time
from decimal import Decimal, InvalidOperation

import numpy as np
import pandas as pd
from rail_connectors.connection import connect

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}'.replace(',', ' '))


def normalize_inn_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip()
    s = re.sub(r'\.0$', '', s)
    s = re.sub(r'\D+', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = s.zfill(10)
    elif len(s) == 11:
        s = s.zfill(12)
    return s if len(s) in (10, 12) else None


def normalize_agr_q1(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace('\xa0', '').replace(' ', '').replace(',', '.')
    if s in {'', 'nan', 'None'}:
        return None
    try:
        d = Decimal(s)
        if d == d.to_integral_value():
            return str(int(d))
    except (InvalidOperation, ValueError):
        pass
    s = re.sub(r'\.0$', '', s)
    return s if s not in {'', 'nan', 'None'} else None

In [ ]:
# === Таблицы ===
mrc_table = 'ods_alpha.scd1_mrc_pos_rent'
agr_terms_table = 'ods_alpha.scd1_agr_terms'
agreements_table = 'ods_alpha.scd1_agreements'
companies_table = 'ods_alpha.scd1_companies'

# === Параметры кейса ===
month_start = '2026-04-01'
month_end = '2026-04-30'
target_agr_id = normalize_agr_q1('413636181589')
target_inn = normalize_inn_q1('2259000869')

# === Подключение ===
impala_db = 'sandbox_ai'
impala_mem_limit = '8g'
impala_user_name = 'Shestopalov-VYur'

print('check month:', month_start, '..', month_end)
print('target agr_id:', target_agr_id)
print('target inn:', target_inn)
print('table:', mrc_table)

In [ ]:
# === Excel (апрель) ===
excel_april_path = '/home/jovyan/documents/Equaring/Data/04_Апрель_2026.xlsx'
excel_header_april = 0  # если заголовок на второй строке, поставьте 1


def pick_col_robust(columns, candidates):
    cols = list(columns)
    norm = lambda s: re.sub(r'\s+', ' ', str(s).replace('\xa0', ' ').strip().lower())
    norm_map = {norm(c): c for c in cols}
    for c in candidates:
        if c in cols:
            return c
        nc = norm(c)
        if nc in norm_map:
            return norm_map[nc]
    return None


def to_num_series(s):
    return pd.to_numeric(
        s.astype(str)
         .str.replace('\xa0', '', regex=False)
         .str.replace(' ', '', regex=False)
         .str.replace(',', '.', regex=False),
        errors='coerce'
    )


print('excel_april_path:', excel_april_path)
print('excel_header_april:', excel_header_april)

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': impala_db},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': impala_user_name}
)
imp._init_connection()


def run_sql(sql_text, step_name='query', mem_limit=impala_mem_limit):
    start_ts = time.perf_counter()
    print(f'[{step_name}] start')
    with imp:
        imp.execute(f"set MEM_LIMIT={mem_limit}")
        df = imp.fetch(sql_text)
    elapsed = round(time.perf_counter() - start_ts, 2)
    rows = len(df) if isinstance(df, pd.DataFrame) else 0
    print(f'[{step_name}] done in {elapsed}s, rows={rows:,}')
    return df


print('Impala connection initialized')

## 1) Проверка доступа к таблице

In [ ]:
sql_access = f"""
select 1 as probe_ok
from {mrc_table}
limit 1
"""

access_ok = False
access_error = None

try:
    access_df = run_sql(sql_access, step_name='access_check')
    access_ok = True
    print('ACCESS_OK: есть SELECT-доступ к таблице')
    display(access_df)
except Exception as exc:
    access_error = f'{type(exc).__name__}: {exc}'
    print('ACCESS_ERROR:', access_error)

access_result_df = pd.DataFrame([
    {'table': mrc_table, 'access_ok': access_ok, 'error': access_error}
])
display(access_result_df)

## 2) Дубли по `c_nmrc + d_rent`

In [ ]:
if not access_ok:
    print('SKIP: нет доступа к таблице.')
else:
    sql_dup_summary = f"""
    with base as (
        select
            cast(c_nmrc as string) as c_nmrc,
            cast(d_rent as date) as d_rent_dt,
            coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
        from {mrc_table}
        where c_nmrc is not null
          and cast(d_rent as date) is not null
    ), active as (
        select *
        from base
        where ods_deleted_flg not in ('1', 'Y', 'y')
    ), grouped as (
        select
            c_nmrc,
            d_rent_dt,
            count(*) as row_cnt
        from active
        group by c_nmrc, d_rent_dt
    )
    select
        (select count(*) from active) as active_rows,
        (select count(*) from grouped) as key_cnt,
        coalesce((select count(*) from grouped where row_cnt > 1), 0) as duplicate_key_cnt,
        coalesce((select sum(row_cnt - 1) from grouped where row_cnt > 1), 0) as duplicate_row_overhead
    """

    dup_summary_df = run_sql(sql_dup_summary, step_name='dup_summary')
    if len(dup_summary_df):
        key_cnt = float(dup_summary_df.loc[0, 'key_cnt'])
        dup_key_cnt = float(dup_summary_df.loc[0, 'duplicate_key_cnt'])
        dup_summary_df['duplicate_key_pct'] = (dup_key_cnt / key_cnt * 100.0) if key_cnt else np.nan
    display(dup_summary_df)

    sql_dup_top = f"""
    with base as (
        select
            cast(c_nmrc as string) as c_nmrc,
            cast(d_rent as date) as d_rent_dt,
            coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
        from {mrc_table}
        where c_nmrc is not null
          and cast(d_rent as date) is not null
    ), active as (
        select *
        from base
        where ods_deleted_flg not in ('1', 'Y', 'y')
    )
    select
        c_nmrc,
        d_rent_dt,
        count(*) as row_cnt
    from active
    group by c_nmrc, d_rent_dt
    having count(*) > 1
    order by row_cnt desc, d_rent_dt desc, c_nmrc
    limit 200
    """

    dup_top_df = run_sql(sql_dup_top, step_name='dup_top_200')
    display(dup_top_df)

## 3) Конфликты: один `c_nmrc + d_rent`, разные `n_amt`

In [ ]:
if not access_ok:
    print('SKIP: нет доступа к таблице.')
else:
    sql_conflict_summary = f"""
    with base as (
        select
            cast(c_nmrc as string) as c_nmrc,
            cast(d_rent as date) as d_rent_dt,
            cast(n_amt as double) as n_amt_num,
            coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
        from {mrc_table}
        where c_nmrc is not null
          and cast(d_rent as date) is not null
    ), active as (
        select *
        from base
        where ods_deleted_flg not in ('1', 'Y', 'y')
    ), grouped as (
        select
            c_nmrc,
            d_rent_dt,
            count(*) as row_cnt,
            count(distinct n_amt_num) as distinct_amt_cnt
        from active
        group by c_nmrc, d_rent_dt
    )
    select
        (select count(*) from grouped) as key_cnt,
        coalesce((select count(*) from grouped where distinct_amt_cnt > 1), 0) as conflict_key_cnt,
        coalesce((select sum(row_cnt) from grouped where distinct_amt_cnt > 1), 0) as rows_in_conflicts
    """

    conflict_summary_df = run_sql(sql_conflict_summary, step_name='conflict_summary')
    if len(conflict_summary_df):
        key_cnt = float(conflict_summary_df.loc[0, 'key_cnt'])
        conflict_cnt = float(conflict_summary_df.loc[0, 'conflict_key_cnt'])
        conflict_summary_df['conflict_key_pct'] = (conflict_cnt / key_cnt * 100.0) if key_cnt else np.nan
    display(conflict_summary_df)

    sql_conflict_top = f"""
    with base as (
        select
            cast(c_nmrc as string) as c_nmrc,
            cast(d_rent as date) as d_rent_dt,
            cast(n_amt as double) as n_amt_num,
            coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
        from {mrc_table}
        where c_nmrc is not null
          and cast(d_rent as date) is not null
    ), active as (
        select *
        from base
        where ods_deleted_flg not in ('1', 'Y', 'y')
    )
    select
        c_nmrc,
        d_rent_dt,
        count(*) as row_cnt,
        count(distinct n_amt_num) as distinct_amt_cnt,
        min(n_amt_num) as min_amt,
        max(n_amt_num) as max_amt
    from active
    group by c_nmrc, d_rent_dt
    having count(distinct n_amt_num) > 1
    order by distinct_amt_cnt desc, row_cnt desc, d_rent_dt desc, c_nmrc
    limit 200
    """

    conflict_top_df = run_sql(sql_conflict_top, step_name='conflict_top_200')
    display(conflict_top_df)

    sql_conflict_rows = f"""
    with base as (
        select
            cast(c_nmrc as string) as c_nmrc,
            cast(d_rent as date) as d_rent_dt,
            cast(n_amt as double) as n_amt_num,
            cast(ods_commit_ts as timestamp) as ods_commit_ts,
            cast(ods_insert_ts as timestamp) as ods_insert_ts,
            cast(ods_op_csn as decimal(38, 0)) as ods_op_csn,
            cast(ods_op_type as string) as ods_op_type,
            coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
        from {mrc_table}
        where c_nmrc is not null
          and cast(d_rent as date) is not null
    ), active as (
        select *
        from base
        where ods_deleted_flg not in ('1', 'Y', 'y')
    ), conflict_keys as (
        select c_nmrc, d_rent_dt
        from active
        group by c_nmrc, d_rent_dt
        having count(distinct n_amt_num) > 1
    )
    select a.*
    from active a
    join conflict_keys k
      on k.c_nmrc = a.c_nmrc
     and k.d_rent_dt = a.d_rent_dt
    order by a.d_rent_dt desc, a.c_nmrc, a.n_amt_num desc, a.ods_commit_ts desc
    limit 500
    """

    conflict_rows_df = run_sql(sql_conflict_rows, step_name='conflict_rows_500')
    display(conflict_rows_df)

## 4) Кейс за апрель: `agr_id=413636181589`, `inn=2259000869`

In [ ]:
if not access_ok:
    print('SKIP: нет доступа к таблице.')
else:
    target_agr_sql = str(target_agr_id).replace("'", "''")
    target_inn_sql = str(target_inn).replace("'", "''")

    base_case_cte = f"""
    with target_agreements as (
        select distinct
            cast(a.n_agr as string) as n_agr,
            cast(a.abs_agr_id as string) as agr_id,
            regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn
        from {agreements_table} a
        join {companies_table} c
          on cast(c.n_cmp as string) = cast(a.n_cmp_client as string)
        where coalesce(cast(a.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
          and coalesce(cast(c.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
          and cast(a.abs_agr_id as string) = '{target_agr_sql}'
          and regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') = '{target_inn_sql}'
          and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
          and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
    ), target_terms as (
        select distinct
            ta.n_agr,
            cast(t.c_nmrc as string) as c_nmrc
        from target_agreements ta
        join {agr_terms_table} t
          on cast(t.n_agr as string) = ta.n_agr
         and coalesce(cast(t.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
         and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
         and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
         and t.c_nmrc is not null
    ), rent_base as (
        select
            cast(c_nmrc as string) as c_nmrc,
            cast(d_rent as date) as d_rent_dt,
            cast(n_amt as double) as n_amt_num,
            cast(ods_commit_ts as timestamp) as ods_commit_ts,
            cast(ods_insert_ts as timestamp) as ods_insert_ts,
            cast(ods_op_csn as decimal(38, 0)) as ods_op_csn,
            coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
        from {mrc_table}
        where cast(d_rent as date) between cast('{month_start}' as date) and cast('{month_end}' as date)
          and c_nmrc is not null
    ), rent_ranked as (
        select
            *,
            row_number() over (
                partition by c_nmrc, d_rent_dt
                order by coalesce(ods_commit_ts, ods_insert_ts) desc, ods_op_csn desc, ods_insert_ts desc
            ) as rn
        from rent_base
        where ods_deleted_flg not in ('1', 'Y', 'y')
    ), rent_dedup as (
        select c_nmrc, d_rent_dt, n_amt_num
        from rent_ranked
        where rn = 1
    )
    """

    sql_case_summary = f"""
    {base_case_cte}
    select
        (select count(*) from target_agreements) as active_agreements_cnt,
        (select count(distinct n_agr) from target_agreements) as n_agr_cnt,
        (select count(distinct c_nmrc) from target_terms) as nmrc_from_terms_cnt,
        coalesce((select count(*) from rent_dedup r join target_terms t on t.c_nmrc = r.c_nmrc), 0) as rent_rows_cnt,
        coalesce((select sum(r.n_amt_num) from rent_dedup r join target_terms t on t.c_nmrc = r.c_nmrc), 0) as commission_monthly_april
    """

    case_summary_df = run_sql(sql_case_summary, step_name='case_summary_april')
    display(case_summary_df)

    sql_case_nmrc = f"""
    {base_case_cte}
    select
        ta.agr_id,
        ta.inn,
        tt.n_agr,
        tt.c_nmrc
    from target_terms tt
    join target_agreements ta
      on ta.n_agr = tt.n_agr
    order by tt.c_nmrc
    """

    case_nmrc_df = run_sql(sql_case_nmrc, step_name='case_nmrc_list')
    display(case_nmrc_df)

    sql_case_rent_details = f"""
    {base_case_cte}
    select
        ta.agr_id,
        ta.inn,
        tt.n_agr,
        r.c_nmrc,
        r.d_rent_dt,
        r.n_amt_num
    from rent_dedup r
    join target_terms tt
      on tt.c_nmrc = r.c_nmrc
    join target_agreements ta
      on ta.n_agr = tt.n_agr
    order by r.d_rent_dt, r.c_nmrc
    """

    case_rent_details_df = run_sql(sql_case_rent_details, step_name='case_rent_details_april')
    display(case_rent_details_df)

## Как читать результат

- **Пункт 1:** `access_ok=True` значит доступ есть.
- **Пункт 2:** `duplicate_key_cnt > 0` значит есть дубли по `c_nmrc + d_rent`.
- **Пункт 3:** `conflict_key_cnt > 0` значит есть конфликтные суммы `n_amt` по одному ключу.
- **Пункт 4:**
  - `nmrc_from_terms_cnt` показывает, найдена ли связка `agr_id+inn -> c_nmrc` за апрель;
  - `commission_monthly_april` — сумма `n_amt` за апрель после дедупликации;
  - детальная таблица показывает все строки, вошедшие в сумму.

## 5) Апрельский маппинг `c_nmrc -> inn+agr_id` и coverage

In [ ]:
if not access_ok:
    print('SKIP: нет доступа к таблице.')
else:
    april_mapping_cte = f"""
    with rent_base as (
        select
            cast(c_nmrc as string) as c_nmrc,
            cast(d_rent as date) as d_rent_dt,
            cast(n_amt as double) as n_amt_num,
            cast(ods_commit_ts as timestamp) as ods_commit_ts,
            cast(ods_insert_ts as timestamp) as ods_insert_ts,
            cast(ods_op_csn as decimal(38, 0)) as ods_op_csn,
            coalesce(cast(ods_deleted_flg as string), '0') as ods_deleted_flg
        from {mrc_table}
        where c_nmrc is not null
          and cast(d_rent as date) between cast('{month_start}' as date) and cast('{month_end}' as date)
    ), rent_ranked as (
        select
            *,
            row_number() over (
                partition by c_nmrc, d_rent_dt
                order by coalesce(ods_commit_ts, ods_insert_ts) desc, ods_op_csn desc, ods_insert_ts desc
            ) as rn
        from rent_base
        where ods_deleted_flg not in ('1', 'Y', 'y')
    ), rent_dedup as (
        select c_nmrc, d_rent_dt, n_amt_num
        from rent_ranked
        where rn = 1
    ), terms_active as (
        select distinct
            cast(t.n_agr as string) as n_agr,
            cast(t.c_nmrc as string) as c_nmrc,
            cast(t.d_valid_from as date) as d_valid_from,
            cast(t.d_valid_to as date) as d_valid_to
        from {agr_terms_table} t
        where coalesce(cast(t.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
          and t.c_nmrc is not null
          and cast(t.d_valid_from as date) <= cast('{month_end}' as date)
          and (t.d_valid_to is null or cast(t.d_valid_to as date) >= cast('{month_start}' as date))
    ), agreements_active as (
        select distinct
            cast(a.n_agr as string) as n_agr,
            cast(a.abs_agr_id as string) as agr_id,
            cast(a.n_cmp_client as string) as n_cmp_client,
            cast(a.d_valid_from as date) as d_valid_from,
            cast(a.d_valid_to as date) as d_valid_to
        from {agreements_table} a
        where coalesce(cast(a.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
          and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
          and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
    ), companies_active as (
        select distinct
            cast(c.n_cmp as string) as n_cmp,
            regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn_key
        from {companies_table} c
        where coalesce(cast(c.ods_deleted_flg as string), '0') not in ('1', 'Y', 'y')
    ), mapped_raw as (
        select
            r.c_nmrc,
            r.d_rent_dt,
            r.n_amt_num,
            c.inn_key,
            cast(a.agr_id as string) as agr_id_key
        from rent_dedup r
        left join terms_active t
          on t.c_nmrc = r.c_nmrc
         and r.d_rent_dt between t.d_valid_from and coalesce(t.d_valid_to, cast('2999-12-31' as date))
        left join agreements_active a
          on a.n_agr = t.n_agr
         and r.d_rent_dt between a.d_valid_from and coalesce(a.d_valid_to, cast('2999-12-31' as date))
        left join companies_active c
          on c.n_cmp = a.n_cmp_client
    )
    """

    sql_mapping_coverage = f"""
    {april_mapping_cte}
    , map_stats as (
        select
            c_nmrc,
            d_rent_dt,
            n_amt_num,
            count(distinct case when inn_key is not null and agr_id_key is not null then concat_ws('|', inn_key, agr_id_key) end) as valid_key_cnt
        from mapped_raw
        group by c_nmrc, d_rent_dt, n_amt_num
    )
    select
        count(*) as rent_rows_after_dedup,
        sum(n_amt_num) as total_n_amt,
        sum(case when valid_key_cnt = 0 then 1 else 0 end) as no_mapping_rows,
        sum(case when valid_key_cnt = 0 then n_amt_num else 0 end) as no_mapping_n_amt,
        sum(case when valid_key_cnt = 1 then 1 else 0 end) as unique_mapping_rows,
        sum(case when valid_key_cnt = 1 then n_amt_num else 0 end) as unique_mapping_n_amt,
        sum(case when valid_key_cnt > 1 then 1 else 0 end) as ambiguous_mapping_rows,
        sum(case when valid_key_cnt > 1 then n_amt_num else 0 end) as ambiguous_mapping_n_amt
    from map_stats
    """

    mapping_coverage_df = run_sql(sql_mapping_coverage, step_name='mapping_coverage_april')
    if len(mapping_coverage_df):
        total_rows = float(mapping_coverage_df.loc[0, 'rent_rows_after_dedup'])
        total_amt = float(mapping_coverage_df.loc[0, 'total_n_amt']) if pd.notna(mapping_coverage_df.loc[0, 'total_n_amt']) else np.nan

        if total_rows:
            mapping_coverage_df['no_mapping_rows_pct'] = mapping_coverage_df['no_mapping_rows'] / total_rows * 100.0
            mapping_coverage_df['unique_mapping_rows_pct'] = mapping_coverage_df['unique_mapping_rows'] / total_rows * 100.0
            mapping_coverage_df['ambiguous_mapping_rows_pct'] = mapping_coverage_df['ambiguous_mapping_rows'] / total_rows * 100.0

        if pd.notna(total_amt) and total_amt != 0:
            mapping_coverage_df['no_mapping_n_amt_pct'] = mapping_coverage_df['no_mapping_n_amt'] / total_amt * 100.0
            mapping_coverage_df['unique_mapping_n_amt_pct'] = mapping_coverage_df['unique_mapping_n_amt'] / total_amt * 100.0
            mapping_coverage_df['ambiguous_mapping_n_amt_pct'] = mapping_coverage_df['ambiguous_mapping_n_amt'] / total_amt * 100.0

    display(mapping_coverage_df)

    sql_mapping_ambiguous_top = f"""
    {april_mapping_cte}
    , map_stats as (
        select
            c_nmrc,
            d_rent_dt,
            n_amt_num,
            count(distinct case when inn_key is not null and agr_id_key is not null then concat_ws('|', inn_key, agr_id_key) end) as valid_key_cnt
        from mapped_raw
        group by c_nmrc, d_rent_dt, n_amt_num
    )
    select *
    from map_stats
    where valid_key_cnt > 1
    order by valid_key_cnt desc, d_rent_dt desc, c_nmrc
    limit 200
    """

    mapping_ambiguous_top_df = run_sql(sql_mapping_ambiguous_top, step_name='mapping_ambiguous_top_200')
    display(mapping_ambiguous_top_df)

## 6) Excel (апрель) vs Lake на уровне `inn+agr_id`

In [ ]:
if not access_ok:
    print('SKIP: нет доступа к таблице.')
else:
    sql_lake_key_april = f"""
    {april_mapping_cte}
    , map_stats as (
        select
            c_nmrc,
            d_rent_dt,
            n_amt_num,
            count(distinct case when inn_key is not null and agr_id_key is not null then concat_ws('|', inn_key, agr_id_key) end) as valid_key_cnt
        from mapped_raw
        group by c_nmrc, d_rent_dt, n_amt_num
    ), mapped_unique as (
        select
            mr.c_nmrc,
            mr.d_rent_dt,
            mr.n_amt_num,
            max(mr.inn_key) as inn_key,
            max(mr.agr_id_key) as agr_id_key
        from mapped_raw mr
        join map_stats ms
          on ms.c_nmrc = mr.c_nmrc
         and ms.d_rent_dt = mr.d_rent_dt
         and ms.n_amt_num = mr.n_amt_num
        where ms.valid_key_cnt = 1
          and mr.inn_key is not null
          and mr.agr_id_key is not null
        group by mr.c_nmrc, mr.d_rent_dt, mr.n_amt_num
    )
    select
        '2026-04' as month_label,
        inn_key,
        agr_id_key,
        sum(n_amt_num) as commission_monthly_lake
    from mapped_unique
    group by inn_key, agr_id_key
    """

    lake_april_key_df = run_sql(sql_lake_key_april, step_name='lake_april_key_agg')
    display(lake_april_key_df.head(50))

    ex_raw = pd.read_excel(excel_april_path, header=excel_header_april)

    excel_col_map = {
        'inn_col': ['ИНН', 'inn', 'c_inn'],
        'agr_col': ['ID договора', 'Номер договора', 'agr_id', 'abs_agr_id'],
        'comm_monthly_col': [
            'Комиссия в месяц',
            'Комиссия CN (₽ в месяц)',
            'Комиссия (₽ в месяц)',
            'Комиссия \n(₽ в месяц)',
            'Комиссия (руб в месяц)'
        ],
    }

    resolved_excel_cols = {k: pick_col_robust(ex_raw.columns, v) for k, v in excel_col_map.items()}
    missing_excel_cols = [k for k, v in resolved_excel_cols.items() if v is None]
    if missing_excel_cols:
        raise ValueError(f'Не найдены колонки Excel: {missing_excel_cols}. Доступные: {list(ex_raw.columns)}')

    ex = ex_raw.copy()
    ex['inn_key'] = ex[resolved_excel_cols['inn_col']].apply(normalize_inn_q1)
    ex['agr_id_key'] = ex[resolved_excel_cols['agr_col']].apply(normalize_agr_q1)
    ex['commission_monthly_excel'] = to_num_series(ex[resolved_excel_cols['comm_monthly_col']])
    ex['month_label'] = '2026-04'

    excel_april_key_df = (
        ex.dropna(subset=['inn_key', 'agr_id_key'])
          .groupby(['month_label', 'inn_key', 'agr_id_key'], as_index=False)
          .agg({'commission_monthly_excel': 'max'})
    )

    print('Excel resolved columns:', resolved_excel_cols)
    display(excel_april_key_df.head(50))

    compare_april_key_df = lake_april_key_df.merge(
        excel_april_key_df,
        on=['month_label', 'inn_key', 'agr_id_key'],
        how='outer',
        indicator=True,
    )
    compare_april_key_df = compare_april_key_df.rename(columns={'_merge': 'merge_status'})

    compare_april_key_df['commission_monthly_lake'] = compare_april_key_df['commission_monthly_lake'].fillna(0.0)
    compare_april_key_df['commission_monthly_excel'] = compare_april_key_df['commission_monthly_excel'].fillna(0.0)
    compare_april_key_df['delta_abs'] = (
        compare_april_key_df['commission_monthly_lake'] - compare_april_key_df['commission_monthly_excel']
    )
    compare_april_key_df['delta_pct'] = np.where(
        compare_april_key_df['commission_monthly_excel'] != 0,
        compare_april_key_df['delta_abs'].abs() / compare_april_key_df['commission_monthly_excel'].abs() * 100.0,
        np.nan,
    )

    compare_april_key_df = compare_april_key_df.sort_values(
        by='delta_abs', key=lambda s: s.abs(), ascending=False
    )

    display(compare_april_key_df.head(200))

    intersection_df = compare_april_key_df[compare_april_key_df['merge_status'] == 'both'].copy()

    month_total_compare_df = pd.DataFrame([
        {
            'month_label': '2026-04',
            'excel_total': float(excel_april_key_df['commission_monthly_excel'].fillna(0).sum()) if len(excel_april_key_df) else 0.0,
            'lake_total_unique_mapping': float(lake_april_key_df['commission_monthly_lake'].fillna(0).sum()) if len(lake_april_key_df) else 0.0,
            'intersection_excel_total': float(intersection_df['commission_monthly_excel'].fillna(0).sum()) if len(intersection_df) else 0.0,
            'intersection_lake_total': float(intersection_df['commission_monthly_lake'].fillna(0).sum()) if len(intersection_df) else 0.0,
            'excel_key_cnt': int(len(excel_april_key_df)),
            'lake_key_cnt': int(len(lake_april_key_df)),
            'intersection_key_cnt': int(len(intersection_df)),
        }
    ])

    month_total_compare_df['total_delta_abs'] = (
        month_total_compare_df['lake_total_unique_mapping'] - month_total_compare_df['excel_total']
    )
    month_total_compare_df['total_delta_pct'] = np.where(
        month_total_compare_df['excel_total'] != 0,
        month_total_compare_df['total_delta_abs'].abs() / month_total_compare_df['excel_total'].abs() * 100.0,
        np.nan,
    )

    month_total_compare_df['intersection_delta_abs'] = (
        month_total_compare_df['intersection_lake_total'] - month_total_compare_df['intersection_excel_total']
    )
    month_total_compare_df['intersection_delta_pct'] = np.where(
        month_total_compare_df['intersection_excel_total'] != 0,
        month_total_compare_df['intersection_delta_abs'].abs() / month_total_compare_df['intersection_excel_total'].abs() * 100.0,
        np.nan,
    )

    merge_status_stats_df = (
        compare_april_key_df.groupby('merge_status', as_index=False)
        .agg(
            key_cnt=('inn_key', 'count'),
            excel_sum=('commission_monthly_excel', 'sum'),
            lake_sum=('commission_monthly_lake', 'sum'),
        )
    )

    print('Month totals:')
    display(month_total_compare_df)
    print('Coverage by merge status:')
    display(merge_status_stats_df)

## 7) Drilldown по кейсу `agr_id=413636181589`, `inn=2259000869` (апрель)

In [ ]:
if not access_ok:
    print('SKIP: нет доступа к таблице.')
else:
    if 'compare_april_key_df' not in globals():
        raise RuntimeError('Сначала запустите секцию 6 (Excel vs Lake compare).')

    case_key_df = compare_april_key_df[
        (compare_april_key_df['inn_key'] == target_inn)
        & (compare_april_key_df['agr_id_key'] == target_agr_id)
    ].copy()

    if case_key_df.empty:
        case_key_df = pd.DataFrame([
            {
                'month_label': '2026-04',
                'inn_key': target_inn,
                'agr_id_key': target_agr_id,
                'commission_monthly_lake': 0.0,
                'commission_monthly_excel': 0.0,
                'delta_abs': 0.0,
                'delta_pct': np.nan,
                'merge_status': 'not_found_in_both',
            }
        ])

    print('Case compare (key-level):')
    display(case_key_df[['month_label', 'inn_key', 'agr_id_key', 'commission_monthly_excel', 'commission_monthly_lake', 'delta_abs', 'delta_pct', 'merge_status']])

    if 'case_rent_details_df' in globals() and len(case_rent_details_df):
        case_nmrc_rollup_df = (
            case_rent_details_df
            .groupby(['agr_id', 'inn', 'n_agr', 'c_nmrc'], as_index=False)
            .agg(commission_monthly_n_amt=('n_amt_num', 'sum'))
            .sort_values('commission_monthly_n_amt', ascending=False)
        )
        print('Case c_nmrc-level details (from section 4):')
        display(case_nmrc_rollup_df)
    else:
        print('Нет детализации по c_nmrc: запустите секцию 4.')

## Рекомендуемый порядок запуска

1. Секции 1-4 (доступ, дубли, конфликты, базовый кейс).
2. Секция 5 (coverage маппинга `c_nmrc -> inn+agr_id`).
3. Секция 6 (сверка Excel vs Lake на уровне `inn+agr_id`).
4. Секция 7 (итоговый drilldown по кейсу с фото).

Ключевая метрика для сопоставимости:
- `intersection_delta_pct` из `month_total_compare_df` (чистое сравнение сумм на пересечении ключей);
- отдельно анализируйте `no_mapping_n_amt_pct` и `ambiguous_mapping_n_amt_pct` как качество покрытия маппинга.